# 2 · Geometry optimisation

In [ ]:
# Install packages (only needed on Google Colab)
!pip install -q ase pyscf py3Dmol

## PySCF as an ASE calculator

ASE works in eV and Å, PySCF in Hartree and Bohr. This class converts between them.

In [ ]:
from ase.calculators.calculator import Calculator, all_changes
from ase.units import Hartree, Bohr
from pyscf import gto, scf


class PySCFCalculator(Calculator):
    """HF energies and forces from PySCF."""

    implemented_properties = ["energy", "forces"]

    def __init__(self, basis="def2-svp", **kwargs):
        super().__init__(**kwargs)
        self.basis = basis

    def calculate(self, atoms=None, properties=["energy"], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)
        mol = gto.M(
            atom=[(s, tuple(p)) for s, p in zip(self.atoms.get_chemical_symbols(), self.atoms.positions)],
            basis=self.basis,
            unit="Angstrom",
            verbose=0,
        )
        mf = scf.RHF(mol).run()
        gradient = mf.nuc_grad_method().kernel()  # Hartree/Bohr
        self.results["energy"] = mf.e_tot * Hartree
        self.results["forces"] = -gradient * Hartree / Bohr

## Viewing structures

Shows two structures side by side (left: start, right: optimised). Rotation and zoom are linked.

In [ ]:
from io import StringIO
from ase.io import write
import py3Dmol


def to_xyz(atoms):
    out = StringIO()
    write(out, atoms, format="xyz")
    return out.getvalue()


def show_before_after(before, after):
    view = py3Dmol.view(width=800, height=350, viewergrid=(1, 2), linked=True)
    view.addModel(to_xyz(before), "xyz", viewer=(0, 0))
    view.addModel(to_xyz(after), "xyz", viewer=(0, 1))
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.zoomTo()
    view.show()

## Part A: N₂ (HF/def2-SVP)

Start from a stretched bond.

In [ ]:
from io import StringIO
from ase.io import read, write
from ase.optimize import BFGS

xyz = """2
N2, stretched
N   0.000   0.000   0.000
N   0.000   0.000   1.400
"""

n2 = read(StringIO(xyz), format="xyz")
print(f"Start: N-N = {n2.get_distance(0, 1):.3f} Å")

In [ ]:
n2_start = n2.copy()

n2.calc = PySCFCalculator()
opt = BFGS(n2, trajectory="n2_opt.traj")
opt.run(fmax=0.01)  # eV/Å

print(f"Final: N-N = {n2.get_distance(0, 1):.3f} Å")
print(f"E = {n2.get_potential_energy() / Hartree:.6f} Hartree")

In [ ]:
show_before_after(n2_start, n2)

In [ ]:
import matplotlib.pyplot as plt

trajectory = read("n2_opt.traj", index=":")
plt.plot([image.get_distance(0, 1) for image in trajectory], "o-")
plt.xlabel("Step")
plt.ylabel("N–N distance (Å)")
plt.show()

## Part B: aspirin (HF/STO-3G)

For a larger molecule we use HF with a minimal basis set (STO-3G), which is much faster. Start structure from PubChem.

In [ ]:
xyz = """21
aspirin (PubChem CID 2244)
O      1.2333     0.5540     0.7792
O     -0.6952    -2.7148    -0.7502
O      0.7958    -2.1843     0.8685
O      1.7813     0.8105    -1.4821
C     -0.0857     0.6088     0.4403
C     -0.7927    -0.5515     0.1244
C     -0.7288     1.8464     0.4133
C     -2.1426    -0.4741    -0.2184
C     -2.0787     1.9238     0.0706
C     -2.7855     0.7636    -0.2453
C     -0.1409    -1.8536     0.1477
C      2.1094     0.6715    -0.3113
C      3.5305     0.5996     0.1635
H     -0.1851     2.7545     0.6593
H     -2.7247    -1.3605    -0.4564
H     -2.5797     2.8872     0.0506
H     -3.8374     0.8238    -0.5090
H      3.7290     1.4184     0.8593
H      4.2045     0.6969    -0.6924
H      3.7105    -0.3659     0.6426
H     -0.2555    -3.5916    -0.7337
"""

aspirin = read(StringIO(xyz), format="xyz")
print(aspirin.get_chemical_formula())

In [ ]:
aspirin_start = aspirin.copy()

aspirin.calc = PySCFCalculator(basis="sto-3g")
opt = BFGS(aspirin, trajectory="aspirin_opt.traj")
opt.run(fmax=0.01)  # eV/Å

print(f"E = {aspirin.get_potential_energy() / Hartree:.6f} Hartree")

In [ ]:
show_before_after(aspirin_start, aspirin)

In [ ]:
print(to_xyz(aspirin))